[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/thermodynamics/entropy_production/entropy_production.ipynb)

# Entropy Production in Heat Exchange

This example computes how two insulated bodies at different temperatures share their energy until their temperatures meet. Heat flows from the hotter body to the colder one, which keeps the total energy fixed and produces entropy, because the heat leaves at a higher temperature than it arrives at; the bodies come to rest at the state of most entropy their total energy allows.

A state is modelled as the two bodies' energies, an antivector of the plane spanned by `mv.a.dual()` and `mv.b.dual()`. The entropy's gradient, each body's received heat over its temperature, `mv.a / temperature_a + mv.b / temperature_b`, pairs with a change of state to the entropy it produces. The mobility is a map from gradients to changes of state, a dyad, `transfer * (Gradient & transfer)` times a positive strength: it sends every gradient into the direction `transfer` that moves energy from one body to the other. It sends the entropy's gradient to the heat flow, and the total energy's gradient, `mv.a + mv.b`, which reads `transfer` as nothing, to nothing; so the flow keeps the energy and, the dyad being positive, never lowers the entropy.

In the notation of differential equations, with a dot for the rate of change in time, the bodies follow Newton's law of cooling, $C_a \dot T_a = -k(T_a - T_b) = -C_b \dot T_b$, and the entropy is produced at the rate $\dot S = k (T_a - T_b)^2 / (T_a T_b) \ge 0$. In the matrix notation of the GENERIC formulation this dissipative part is $\dot x = M\,dS$, with $M$ positive and $M\,dE = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples,jax] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import Algebra, stack
from numga.backend.jax import JaxContext, derivative
from examples.util.animation import show_animation
from examples.thermodynamics.entropy_production import render

ga = Algebra("a0b0")
mv = JaxContext(ga, np.float64).multivector
Scalar = ga.gatype.scalar()                               # Scalar+
State = ga.gatype.antivector()                            # Antivector-
Gradient = ga.gatype.vector()                             # Vector+
Mobility = ga.gatype((State, Gradient))                   # State- <- Gradient+

# The total energy's gradient, and the direction moving energy from body a to body b, which it reads as nothing.
energy_gradient = mv.a + mv.b                             # [] Gradient+
transfer = mv.b.dual() - mv.a.dual()                      # [] State-
# The bodies' heat capacities, and the conductance of the bar between them.
capacity_a, capacity_b, conductance = 1.0, 2.0, 0.6

## The entropy, its gradient and the mobility

A body's temperature is its energy over its heat capacity, and its entropy the heat capacity times the logarithm of its temperature. The mobility's strength is the conductance times the product of the temperatures, because that makes the heat flow the conductance times the temperature difference, as Newton's law of cooling has it. Energies are in units of the heat capacity times a reference temperature, temperatures in that reference. Body a starts hot and body b cold; the first chart shows the plane of their energies, the entropy's level lines in teal and the line of fixed total energy, along which the state can only move; the second the entropy along that line; the third the two temperatures over time.

In [ ]:
temperature_a = (mv.a & State) / capacity_a                                                 # [] Scalar+ <- State-
temperature_b = (mv.b & State) / capacity_b                                                 # [] Scalar+ <- State-
# The mobility of unit strength: it sends every gradient along the transfer, by the gradient's reading of it.
unit_mobility = transfer * (Gradient & transfer)                                            # [] State- <- Gradient+

def entropy(state: State) -> Scalar:
    return capacity_a * temperature_a(state).log() + capacity_b * temperature_b(state).log()

def entropy_gradient(state: State) -> Gradient:
    """Each body's received heat over its temperature."""
    return mv.a / temperature_a(state) + mv.b / temperature_b(state)

def mobility(state: State, strength: float) -> Mobility:
    """The unit mobility, with the strength times the temperatures' product."""
    return unit_mobility * (strength * temperature_a(state) * temperature_b(state))

initial_temperature_a, initial_temperature_b = 2.8, 0.6
start = (mv.a.dual() * (capacity_a * initial_temperature_a)
         + mv.b.dual() * (capacity_b * initial_temperature_b))                              # [] State-
heat_flow = mobility(start, conductance)(entropy_gradient(start))                           # [] State-
temperatures = stack((temperature_a, temperature_b))                                        # [bodies] Scalar+ <- State-
capacities = np.array([capacity_a, capacity_b])
render.draw(entropy, temperatures, transfer, capacities, start, heat_flow);

In [ ]:
print("heat flow:", (mv.a & heat_flow).to_array(), (mv.b & heat_flow).to_array())
print("entropy produced per unit time:", (entropy_gradient(start) & heat_flow).to_array())

## The exchange over time

The heat flow is the conductance times the temperature difference, and that difference decays exponentially with the relaxation time `1 / (conductance * (1 / capacity_a + 1 / capacity_b))`. So a step of any length is exact as one application of the mobility with the conductance replaced by its integral over the step. The animation runs over several relaxation times.

In [ ]:
inverse_capacity = 1 / capacity_a + 1 / capacity_b
relaxation_time = 1 / (conductance * inverse_capacity)
n_frames, n_relaxation_times = 72, 5
dt = n_relaxation_times * relaxation_time / (n_frames - 1)
integrated_conductance = -np.expm1(-conductance * inverse_capacity * dt) / inverse_capacity

def exchange():
    """The bodies' state, its heat flow and the time at each frame."""
    state = start
    for frame in range(n_frames):
        yield state, mobility(state, conductance)(entropy_gradient(state)), frame * dt
        state = state + mobility(state, integrated_conductance)(entropy_gradient(state))

show_animation(render.exchange(entropy, temperatures, transfer, capacities, exchange()), "entropy_production",
               duration_ms=75)

The bodies are coloured by their gap to the common temperature, half by the gap and half by its logarithm, so that the closing gap keeps changing their colours to the end. The hot body a gives energy to body b along the line of fixed total energy, the arrow shrinking as the temperatures approach, and the state climbs the entropy's hill, while the two temperatures close on the common one. It comes to rest where the black level line of the entropy touches the line of fixed energy, at the top of the hill, where both bodies are at the common temperature $4/3$, the total energy over the total heat capacity.

In [ ]:
end = list(exchange())[-1][0]                                                               # [] State-
final_temperatures = temperatures(end)                                                      # [bodies] Scalar+

In [ ]:
print("final temperatures:", final_temperatures.to_array())
print("relaxation time:", relaxation_time)

In [ ]:
# checks
states = [state for state, _, _ in exchange()]
np.testing.assert_allclose([((energy_gradient & state) - (energy_gradient & start)).to_array() for state in states], 0, atol=1e-10)
assert np.all(np.diff([entropy(state).to_array() for state in states]) >= -1e-12)
# The entropy's gradient, as a reading of changes of state, is the entropy's derivative.
np.testing.assert_allclose(((entropy_gradient(start) & State) - derivative(entropy)(start)).kernel, 0, atol=1e-10)
np.testing.assert_allclose(mobility(start, conductance)(energy_gradient).kernel, 0, atol=1e-10)
np.testing.assert_allclose(final_temperatures.to_array(), 4 / 3, atol=1e-2)